[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_sanity_check.ipynb)

# MeanFlow sanity check — FashionMNIST + tiny DiT

이 임시 노트북의 목적은 **논문 분석 실험 전에 현재 tiny DiT + MeanFlow 조합 자체가 실제로 학습되는지 확인**하는 것이다.

학습 설정:
- FashionMNIST, \(28\times 28\)
- tiny DiT: patch 4, hidden 96, depth 3, heads 4
- MeanFlow objective: `flow_basic.ipynb`와 같은 linear path와 JVP identity
- 총 3000 step
- 250 step마다 고정 입력 진단

확인 항목:
1. 같은 고정 noise 16개에서 1-step 생성 결과를 PNG로 저장
2. loss / MeanFlow identity residual / gradient norm
3. 고정 validation batch에서 predicted average velocity와 MeanFlow target의 cosine
4. 각 DiT block의 초기 weight 대비 relative drift
5. 각 DiT block hidden feature의 std / effective rank / singular-value spectrum

**이미지는 셀에 표시하지 않는다.**  
`/content/meanflow_sanity/.../samples/` 아래에 step별 PNG로 저장한다.

**그래프와 feature/weight 진단은 TensorBoard에 기록한다.**  
아래 TensorBoard 셀을 **학습 전에 먼저 실행**한다.


## 0. Setup

Colab 런타임은 **T4 GPU**를 대상으로 설정되어 있다.  
노트북 metadata에도 GPU/T4를 지정하고, 아래 셀에서 실제 연결된 GPU 이름을 확인한다.

실험 결과는 한 번 실행할 때마다 별도 `RUN_NAME` 폴더에 저장한다.


In [ ]:
# @title 0-1. Install / imports / run directories
!pip -q install datasets tensorboard

import copy
import math
import os
import random
import time
import warnings
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from torch.func import jvp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.transforms import ToTensor
from torchvision.utils import save_image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("This notebook is configured for a Colab T4 GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

if "T4" not in gpu_name:
    warnings.warn(
        "This run is sized for a T4. Colab connected a different GPU: "
        f"{gpu_name}"
    )

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)

TRAIN_STEPS = 3000
CHECK_EVERY = 250
SCALAR_LOG_EVERY = 10
BATCH_SIZE = 128
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
FIXED_SAMPLE_COUNT = 16
DIAGNOSTIC_BATCH_SIZE = 64
EMA_DECAY = 0.999

ROOT_DIR = "/content/meanflow_sanity"
RUN_NAME = time.strftime("%Y%m%d_%H%M%S")
LOG_ROOT = os.path.join(ROOT_DIR, "tensorboard")
LOG_DIR = os.path.join(LOG_ROOT, RUN_NAME)
RUN_DIR = os.path.join(ROOT_DIR, "runs", RUN_NAME)
SAMPLE_DIR = os.path.join(RUN_DIR, "samples")
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")

for path in [LOG_ROOT, LOG_DIR, SAMPLE_DIR, CHECKPOINT_DIR]:
    os.makedirs(path, exist_ok=True)

writer = SummaryWriter(LOG_DIR)
writer.add_text(
    "run/config",
    (
        f"GPU={gpu_name}, steps={TRAIN_STEPS}, check_every={CHECK_EVERY}, "
        f"batch={BATCH_SIZE}, lr={LEARNING_RATE}"
    ),
    global_step=0,
)
writer.flush()

print("RUN_NAME:", RUN_NAME)
print("TensorBoard root:", LOG_ROOT)
print("Sample folder:", SAMPLE_DIR)


## 1. TensorBoard — **학습 전에 먼저 실행**

이 셀은 학습과 분리되어 있다.

1. 위 Setup 셀 실행
2. **이 TensorBoard 셀 실행**
3. 그 다음 데이터/모델/학습 셀 실행

학습이 시작되면 같은 TensorBoard 화면에서 다음 곡선이 계속 추가된다.

- `train/loss`
- `train/identity_residual`
- `train/target_cosine_batch`
- `train/grad_norm`
- `diagnostic/fixed_target_cosine`
- `diagnostic/r_eq_t_velocity_mse`
- `weights/block_*_relative_drift`
- `features/block_*/std`
- `features/block_*/effective_rank`
- `features/block_*/top_sv_ratio`

`Histograms` 탭에는 각 block의 hidden-feature singular values가 250 step마다 기록된다.


In [ ]:
# @title 1-1. Launch TensorBoard before training
%load_ext tensorboard
%tensorboard --logdir /content/meanflow_sanity/tensorboard --reload_interval 5


## 2. FashionMNIST

빠른 Colab 다운로드를 위해 Hugging Face dataset을 우선 사용하고, 실패하면 명시적인 fallback mirror를 사용한다.

이미지는 \([0,1]\)에서 MeanFlow 학습용 \([-1,1]\) 범위로 바꾼다.


In [ ]:
# @title 2-1. Download FashionMNIST and create loaders
try:
    dataset = load_dataset("zalando-datasets/fashion_mnist")
except Exception as primary_error:
    print("Primary FashionMNIST source failed:", repr(primary_error))
    print("Using fallback mirror: anonyme449/fashion_mnist")
    dataset = load_dataset("anonyme449/fashion_mnist")

to_tensor = ToTensor()


def collate_batch(batch):
    images = torch.stack([to_tensor(item["image"]) for item in batch])
    images = images * 2.0 - 1.0
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )
    return images, labels


train_loader = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=DIAGNOSTIC_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
    collate_fn=collate_batch,
)

train_batch = next(iter(train_loader))
print("train image shape:", train_batch[0].shape)
print("train range:", train_batch[0].min().item(), train_batch[0].max().item())


## 3. tiny DiT

`flow_basic.ipynb`의 공통 tiny DiT 설정을 그대로 사용한다.

- 입력: \(\mathbb R^{1\times28\times28}\)
- patch size 4 → 49 tokens
- hidden dimension 96
- 3 Transformer blocks
- 4 attention heads
- MeanFlow에는 ordered scalar conditions \((r,t)\)를 넣는다.

추가된 것은 **진단용 `forward_features`**뿐이다.  
생성 출력을 만드는 `forward` 경로는 같은 backbone을 사용한다.

PyTorch `MultiheadAttention`의 fast/SDPA 경로는 forward-mode AD(JVP)를 지원하지 않는 경우가 있으므로, 이 sanity notebook에서는 fastpath를 끄고 `need_weights=True`의 명시적 attention 계산 경로를 사용한다. attention weight 자체는 버린다. 이는 MeanFlow JVP를 안정적으로 계산하기 위한 구현상의 조치다.


In [ ]:
# @title 3-1. tiny DiT with diagnostic feature access
class ScalarEmbed(nn.Module):
    def __init__(self, dim, fourier=64):
        super().__init__()
        self.fourier = fourier
        self.mlp = nn.Sequential(
            nn.Linear(fourier, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, scalar):
        scalar = scalar.reshape(-1, 1)
        half = self.fourier // 2
        frequencies = torch.exp(
            torch.linspace(
                math.log(1.0),
                math.log(1000.0),
                half,
                device=scalar.device,
            )
        )
        angles = scalar * frequencies[None] * 2.0 * math.pi
        embedding = torch.cat(
            [angles.sin(), angles.cos()],
            dim=1,
        )
        return self.mlp(embedding)


def math_attention_context():
    if DEVICE == "cuda":
        return torch.backends.cuda.sdp_kernel(
            enable_flash=False,
            enable_math=True,
            enable_mem_efficient=False,
        )
    return nullcontext()


class DiTBlock(nn.Module):
    def __init__(self, dim=96, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
        )
        self.attention = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )

    def forward(self, tokens, condition):
        shift1, scale1, gate1, shift2, scale2, gate2 = (
            self.modulation(condition).chunk(6, dim=-1)
        )

        hidden = (
            self.norm1(tokens) * (1.0 + scale1[:, None])
            + shift1[:, None]
        )

        with math_attention_context():
            attended, _ = self.attention(
                hidden,
                hidden,
                hidden,
                need_weights=True,
            )

        tokens = tokens + gate1[:, None] * attended

        hidden = (
            self.norm2(tokens) * (1.0 + scale2[:, None])
            + shift2[:, None]
        )
        tokens = (
            tokens
            + gate2[:, None] * self.feed_forward(hidden)
        )
        return tokens


class TinyDiT(nn.Module):
    def __init__(
        self,
        dim=96,
        depth=3,
        heads=4,
        patch=4,
    ):
        super().__init__()
        self.patch = patch
        self.dim = dim
        self.input_projection = nn.Conv2d(
            1,
            dim,
            kernel_size=patch,
            stride=patch,
        )
        self.position = nn.Parameter(
            torch.randn(1, 49, dim) * 0.02
        )
        self.scalar_embed = ScalarEmbed(dim)

        # Allocate the same scalar-role table as flow_basic.
        self.scalar_roles = nn.Parameter(
            torch.randn(4, dim) * 0.02
        )

        self.blocks = nn.ModuleList(
            [
                DiTBlock(
                    dim=dim,
                    heads=heads,
                )
                for _ in range(depth)
            ]
        )

        self.final = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, patch * patch),
        )

    def _condition(self, scalars):
        if len(scalars) == 0 or len(scalars) > len(self.scalar_roles):
            raise ValueError(
                "Expected between 1 and "
                f"{len(self.scalar_roles)} scalar conditions, "
                f"got {len(scalars)}."
            )

        embeddings = []
        for role_index, scalar in enumerate(scalars):
            embeddings.append(
                self.scalar_embed(scalar)
                + self.scalar_roles[role_index][None]
            )
        return sum(embeddings)

    def _tokens(self, images):
        return (
            self.input_projection(images)
            .flatten(2)
            .transpose(1, 2)
            + self.position
        )

    def forward_features(self, images, *scalars):
        tokens = self._tokens(images)
        condition = self._condition(scalars)

        features = []
        for block in self.blocks:
            tokens = block(tokens, condition)
            features.append(tokens)

        return features

    def forward(self, images, *scalars):
        tokens = self._tokens(images)
        condition = self._condition(scalars)

        for block in self.blocks:
            tokens = block(tokens, condition)

        patches = self.final(tokens).view(
            images.size(0),
            7,
            7,
            self.patch,
            self.patch,
        )

        return (
            patches.permute(0, 1, 3, 2, 4)
            .reshape(images.size(0), 1, 28, 28)
        )


model = TinyDiT().to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("parameters:", parameter_count)


## 4. MeanFlow training objective

`flow_basic.ipynb`와 같은 linear interpolation을 사용한다.

\[
z_t=(1-t)x+t\epsilon,\qquad
v=\epsilon-x,
\]

여기서
- \(x\in\mathbb R^{1\times28\times28}\): FashionMNIST data
- \(\epsilon\sim\mathcal N(0,I)\): noise
- \(t\in[0,1]\)
- \(r\in[0,t]\)

MeanFlow network는

\[
u_\theta:
\mathbb R^{784}\times[0,1]^2\to\mathbb R^{784},
\qquad
(z_t,r,t)\mapsto u_\theta(z_t,r,t)
\]

를 학습한다.

현재 구현의 identity target은

\[
u_{\mathrm{target}}
=
v-(t-r)\frac{d}{dt}u_\theta(z_t,r,t),
\]

이며 total derivative는 trajectory tangent

\[
(\dot z_t,\dot r,\dot t)=(v,0,1)
\]

에 대한 JVP로 계산한다.

따라서 학습 loss와 `identity_residual`은 **같은 residual의 MSE이므로 수치적으로 같은 값**이다. TensorBoard에는 이름을 둘 다 남겨서 무엇을 최적화하는지 명확하게 표시한다.


In [ ]:
# @title 4-1. MeanFlow JVP objective
def sample_meanflow_training_tuple(images):
    batch_size = images.size(0)

    t = torch.rand(
        batch_size,
        device=images.device,
    )
    r = torch.rand(
        batch_size,
        device=images.device,
    ) * t

    noise = torch.randn_like(images)

    t_image = t[:, None, None, None]
    z_t = (
        (1.0 - t_image) * images
        + t_image * noise
    )
    velocity = noise - images

    return z_t, velocity, r, t


def meanflow_identity_outputs(
    current_model,
    z_t,
    velocity,
    r,
    t,
):
    zeros = torch.zeros_like(t)
    ones = torch.ones_like(t)

    def meanflow_function(z_value, r_value, t_value):
        return current_model(
            z_value,
            r_value,
            t_value,
        )

    average_velocity, total_derivative = jvp(
        meanflow_function,
        (z_t, r, t),
        (velocity, zeros, ones),
    )

    interval = (t - r)[:, None, None, None]
    target = (
        velocity
        - interval * total_derivative
    ).detach()

    return average_velocity, target


def meanflow_loss(current_model, images):
    z_t, velocity, r, t = sample_meanflow_training_tuple(images)

    prediction, target = meanflow_identity_outputs(
        current_model,
        z_t,
        velocity,
        r,
        t,
    )

    residual = prediction - target
    loss = residual.square().mean()

    prediction_flat = prediction.flatten(1)
    target_flat = target.flatten(1)

    target_cosine = F.cosine_similarity(
        prediction_flat,
        target_flat,
        dim=1,
        eps=1e-8,
    ).mean()

    return loss, target_cosine


## 5. Fixed inputs for fair progress checks

학습 진행을 비교하려면 매번 다른 입력을 쓰면 안 된다.

그래서 처음 한 번만 다음을 고정한다.

- 생성 확인용 noise 16개
- diagnostic FashionMNIST 64개
- 그 64개에 대응하는 noise
- diagnostic \(t,r\)

250 step마다 **완전히 같은 입력**으로 진단한다.

### 생성 이미지

고정 noise \(z_1\)에서 MeanFlow 1-step 생성은

\[
\hat x_0
=
z_1-u_\theta(z_1,0,1)
\]

으로 계산한다.

이미지는 notebook output에 띄우지 않고 `samples/step_XXXX.png`로 저장한다.


In [ ]:
# @title 5-1. Create fixed samples and fixed diagnostic batch
fixed_generator = torch.Generator().manual_seed(SEED + 100)

fixed_noise = torch.randn(
    FIXED_SAMPLE_COUNT,
    1,
    28,
    28,
    generator=fixed_generator,
).to(DEVICE)

diagnostic_images, _ = next(iter(test_loader))
diagnostic_images = diagnostic_images[
    :DIAGNOSTIC_BATCH_SIZE
].to(
    DEVICE,
    non_blocking=True,
)

diagnostic_noise = torch.randn(
    diagnostic_images.shape,
    generator=fixed_generator,
).to(DEVICE)

diagnostic_t = (
    0.05
    + 0.90
    * torch.rand(
        DIAGNOSTIC_BATCH_SIZE,
        generator=fixed_generator,
    )
).to(DEVICE)

diagnostic_r = (
    torch.rand(
        DIAGNOSTIC_BATCH_SIZE,
        generator=fixed_generator,
    ).to(DEVICE)
    * diagnostic_t
)

initial_block_parameters = []
for block in model.blocks:
    block_snapshot = [
        parameter.detach().clone()
        for parameter in block.parameters()
    ]
    initial_block_parameters.append(block_snapshot)

print("fixed generation noise:", fixed_noise.shape)
print("fixed diagnostic images:", diagnostic_images.shape)


## 6. Diagnostic functions

### A. Target cosine

고정 batch에서

\[
C=
\frac{
\langle u_\theta,u_{\mathrm{target}}\rangle
}{
\|u_\theta\|_2\,
\|u_{\mathrm{target}}\|_2
}
\]

를 계산한다.

그림이 아직 알아보기 어려워도 \(C\)가 상승하면 model output이 학습 target 방향으로 정렬되고 있다는 신호다.

### B. \(r=t\) boundary check

MeanFlow identity에서는 구간 길이가 0이면 average velocity가 instantaneous velocity가 되어야 한다.

\[
u(z_t,t,t)=v.
\]

따라서

\[
E_{\mathrm{boundary}}
=
\mathbb E\|u_\theta(z_t,t,t)-v\|_2^2
\]

를 250 step마다 측정한다.

### C. Weight drift

각 block \(\ell\)의 초기 parameter를 \(\theta_\ell^{(0)}\), 현재 parameter를 \(\theta_\ell^{(k)}\)라 하면

\[
R_\ell(k)
=
\frac{
\|\theta_\ell^{(k)}-\theta_\ell^{(0)}\|_2
}{
\|\theta_\ell^{(0)}\|_2
}.
\]

0에 계속 붙어 있으면 해당 block이 거의 움직이지 않는 것이다.

### D. Hidden feature spectrum

고정된 \(t=0.5,r=0\) input을 각 DiT block에 통과시켜 hidden token matrix를 얻는다.

\[
H_\ell\in\mathbb R^{(B\cdot49)\times96}.
\]

center한 \(H_\ell\)의 singular values를 계산해 다음을 TensorBoard에 남긴다.

- feature std
- singular-value effective rank
- largest singular value / total singular value
- singular values histogram

이 값들은 **좋은 생성 품질의 보증이 아니라, hidden representation이 죽어 있는지/변하고 있는지 확인하는 sanity diagnostic**이다.


In [ ]:
# @title 6-1. Generation, weight, and feature diagnostics
@torch.no_grad()
def save_fixed_generation(current_model, step):
    was_training = current_model.training
    current_model.eval()

    zeros = torch.zeros(
        FIXED_SAMPLE_COUNT,
        device=DEVICE,
    )
    ones = torch.ones(
        FIXED_SAMPLE_COUNT,
        device=DEVICE,
    )

    average_velocity = current_model(
        fixed_noise,
        zeros,
        ones,
    )
    generated = fixed_noise - average_velocity
    generated = generated.clamp(-1.0, 1.0)
    generated = (generated + 1.0) / 2.0

    output_path = os.path.join(
        SAMPLE_DIR,
        f"step_{step:04d}.png",
    )
    save_image(
        generated,
        output_path,
        nrow=4,
        padding=2,
    )

    if was_training:
        current_model.train()

    return output_path


def relative_block_drift(block, initial_parameters):
    numerator = torch.zeros(
        (),
        device=DEVICE,
    )
    denominator = torch.zeros(
        (),
        device=DEVICE,
    )

    for parameter, initial_parameter in zip(
        block.parameters(),
        initial_parameters,
    ):
        current = parameter.detach()
        initial = initial_parameter.to(current.device)

        numerator = numerator + (
            current - initial
        ).square().sum()

        denominator = denominator + (
            initial.square().sum()
        )

    return (
        numerator.sqrt()
        / denominator.sqrt().clamp_min(1e-12)
    ).item()


def effective_rank_from_singular_values(singular_values):
    normalized = (
        singular_values
        / singular_values.sum().clamp_min(1e-12)
    )

    entropy = -(
        normalized
        * normalized.clamp_min(1e-12).log()
    ).sum()

    return entropy.exp()


@torch.no_grad()
def log_fixed_diagnostics(current_model, step):
    was_training = current_model.training
    current_model.eval()

    t_image = diagnostic_t[:, None, None, None]

    z_t = (
        (1.0 - t_image) * diagnostic_images
        + t_image * diagnostic_noise
    )
    velocity = diagnostic_noise - diagnostic_images

    prediction, target = meanflow_identity_outputs(
        current_model,
        z_t,
        velocity,
        diagnostic_r,
        diagnostic_t,
    )

    fixed_target_cosine = F.cosine_similarity(
        prediction.flatten(1),
        target.flatten(1),
        dim=1,
        eps=1e-8,
    ).mean()

    boundary_prediction = current_model(
        z_t,
        diagnostic_t,
        diagnostic_t,
    )
    boundary_mse = F.mse_loss(
        boundary_prediction,
        velocity,
    )

    writer.add_scalar(
        "diagnostic/fixed_target_cosine",
        fixed_target_cosine.item(),
        step,
    )
    writer.add_scalar(
        "diagnostic/r_eq_t_velocity_mse",
        boundary_mse.item(),
        step,
    )

    for block_index, block in enumerate(current_model.blocks):
        drift = relative_block_drift(
            block,
            initial_block_parameters[block_index],
        )
        writer.add_scalar(
            f"weights/block_{block_index}_relative_drift",
            drift,
            step,
        )

    feature_t = torch.full(
        (DIAGNOSTIC_BATCH_SIZE,),
        0.5,
        device=DEVICE,
    )
    feature_r = torch.zeros_like(feature_t)

    feature_z = (
        0.5 * diagnostic_images
        + 0.5 * diagnostic_noise
    )

    block_features = current_model.forward_features(
        feature_z,
        feature_r,
        feature_t,
    )

    feature_summary = {}

    for block_index, features in enumerate(block_features):
        matrix = features.reshape(
            -1,
            features.size(-1),
        ).float()

        matrix = matrix - matrix.mean(
            dim=0,
            keepdim=True,
        )

        singular_values = torch.linalg.svdvals(matrix)
        effective_rank = effective_rank_from_singular_values(
            singular_values
        )

        top_sv_ratio = (
            singular_values[0]
            / singular_values.sum().clamp_min(1e-12)
        )

        feature_std = features.float().std()

        writer.add_scalar(
            f"features/block_{block_index}/std",
            feature_std.item(),
            step,
        )
        writer.add_scalar(
            f"features/block_{block_index}/effective_rank",
            effective_rank.item(),
            step,
        )
        writer.add_scalar(
            f"features/block_{block_index}/top_sv_ratio",
            top_sv_ratio.item(),
            step,
        )
        writer.add_histogram(
            f"features/block_{block_index}/singular_values",
            singular_values.detach().cpu(),
            step,
        )

        feature_summary[f"block_{block_index}_effective_rank"] = (
            effective_rank.item()
        )

    writer.flush()

    if was_training:
        current_model.train()

    return {
        "fixed_target_cosine": fixed_target_cosine.item(),
        "boundary_mse": boundary_mse.item(),
        **feature_summary,
    }


## 7. EMA and training

학습 자체는 한 셀에서 실행한다.

- step 0에서 먼저 고정 noise 생성 PNG와 내부 진단을 저장
- 이후 매 250 step마다 다시 저장/진단
- scalar 학습 로그는 10 step마다 TensorBoard에 기록
- 생성 이미지는 TensorBoard에 넣지 않고 파일로만 저장
- 마지막에는 model/EMA checkpoint를 저장

TensorBoard는 이미 위에서 실행 중이므로 학습 중에도 새 scalar가 같은 화면에 누적된다.


In [ ]:
# @title 7-1. Train 3000 steps and diagnose every 250 steps
class EMA:
    def __init__(self, current_model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(
            current_model
        ).eval()

        for parameter in self.shadow.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, current_model):
        for shadow_parameter, parameter in zip(
            self.shadow.parameters(),
            current_model.parameters(),
        ):
            shadow_parameter.mul_(self.decay).add_(
                parameter,
                alpha=1.0 - self.decay,
            )


def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

ema = EMA(
    model,
    decay=EMA_DECAY,
)

training_iterator = infinite_loader(train_loader)

# Step 0: save exactly the same fixed inputs used later.
step_zero_sample = save_fixed_generation(
    model,
    step=0,
)
step_zero_diagnostics = log_fixed_diagnostics(
    model,
    step=0,
)

print("step 0 sample saved:", step_zero_sample)
print("step 0 diagnostics:", step_zero_diagnostics)

model.train()

for step in range(1, TRAIN_STEPS + 1):
    images, _ = next(training_iterator)
    images = images.to(
        DEVICE,
        non_blocking=True,
    )

    optimizer.zero_grad(
        set_to_none=True,
    )

    loss, batch_target_cosine = meanflow_loss(
        model,
        images,
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP,
    )

    optimizer.step()
    ema.update(model)

    if step == 1 or step % SCALAR_LOG_EVERY == 0:
        writer.add_scalar(
            "train/loss",
            loss.item(),
            step,
        )
        writer.add_scalar(
            "train/identity_residual",
            loss.item(),
            step,
        )
        writer.add_scalar(
            "train/target_cosine_batch",
            batch_target_cosine.item(),
            step,
        )
        writer.add_scalar(
            "train/grad_norm",
            float(grad_norm),
            step,
        )

    if step % CHECK_EVERY == 0:
        sample_path = save_fixed_generation(
            model,
            step,
        )

        diagnostics = log_fixed_diagnostics(
            model,
            step,
        )

        print(
            f"step={step:4d} "
            f"loss={loss.item():.6f} "
            f"fixed_cos={diagnostics['fixed_target_cosine']:.4f} "
            f"boundary_mse={diagnostics['boundary_mse']:.6f} "
            f"sample={os.path.basename(sample_path)}"
        )

writer.flush()

model_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "meanflow_model_final.pt",
)
ema_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "meanflow_ema_final.pt",
)

torch.save(
    model.state_dict(),
    model_checkpoint_path,
)
torch.save(
    ema.shadow.state_dict(),
    ema_checkpoint_path,
)

print("training complete")
print("model checkpoint:", model_checkpoint_path)
print("EMA checkpoint:", ema_checkpoint_path)


## 8. Output locations

생성 이미지는 셀에 표시하지 않는다.

현재 run의 결과는 다음 폴더에 있다.

- `SAMPLE_DIR`: `step_0000.png`, `step_0250.png`, ..., `step_3000.png`
- `LOG_DIR`: TensorBoard event
- `CHECKPOINT_DIR`: final raw model / EMA model

아래 셀은 **이미지를 렌더링하지 않고 파일 이름만 확인**한다.


In [ ]:
# @title 8-1. List saved artifacts without displaying images
sample_files = sorted(
    file_name
    for file_name in os.listdir(SAMPLE_DIR)
    if file_name.endswith(".png")
)

print("RUN_DIR:", RUN_DIR)
print("saved sample files:")
for file_name in sample_files:
    print(" -", file_name)

print("TensorBoard log dir:", LOG_DIR)
print("checkpoint dir:", CHECKPOINT_DIR)
